In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [1]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

# The code demostrates a one sequential program where sections are labelled as
# Alice and Bob. Random choices are generating by measuring the quantum state


In [3]:
simulator = AerSimulator()

In [4]:
# Helper Functions

def quantum_random_bit():
    """Generate one random bit by measuring |+> = (|0> + |1>) / sqrt(2)."""

    # Creates a quantum circuit with 1 qubit and 1 classicall bit
    qc = QuantumCircuit(1, 1)

    # Put the qubit into superposition:
    # |0> becomes |+> = (|0> + |1>) / sqrt(2)
    qc.h(0)

    # Measure the qubit - Gives 0 or 1 with equal probability
    qc.measure(0, 0)

    # Run the circuit once to get one random bit
    result = simulator.run(qc, shots=1, memory=True).result()

    # Extract the measured bit from the simulator result and convert it into INT
    return int(result.get_memory()[0])


def quantum_random_bits(n):
    """Generate n quantum random bits without using Python's random module."""

    # Repeatedly call quantum_random_but() to build a list of n random bits
    return [quantum_random_bit() for _ in range(n)]


def prepare_qubit(bit, basis):
    """
    Alice prepares one qubit.

    basis = 0: computational / rectilinear basis
        bit 0 -> |0>
        bit 1 -> |1>

    basis = 1: diagonal basis
        bit 0 -> |+>
        bit 1 -> |->
    """

    # Start with a fresh qubit in state |0>
    qc = QuantumCircuit(1, 1)

    # If Alice bit is 1, flip |0> to |1>
    # If bit is 0, remain it at |0>
    if bit == 1:
        qc.x(0)

    # If Alice chooses diagonal basis, apply H:
    # |0> becomes |+>
    # |1> becomes |->
    if basis == 1:
        qc.h(0)

    # Return the circuit representing Alice preparing qubit
    return qc


def measure_qubit(qubit_circuit, basis):
    """
    Measure one qubit in the chosen basis.

    basis = 0: computational / rectilinear basis
    basis = 1: diagonal basis
    """

    # Copy the input circuit so the original prepared qubit circuit is not changed
    qc = qubit_circuit.copy()

    # Measure diagonal basis using H before standard mesaurement
    # |+> converts to |0> and |-> converts to |1>
    if basis == 1:
        qc.h(0)

    # Measure the qubit into classical bit
    qc.measure(0, 0)

    # Run the measurement once
    result = simulator.run(qc, shots=1, memory=True).result()

    # Return the measured classical bit as INT
    return int(result.get_memory()[0])


def bit_string(bits):
    """Format a list of bits as a compact string."""

    # Convert each bit to a string and join them together, e.g. [1,0,1] - > "101"
    return ''.join(str(bit) for bit in bits)

In [5]:
# Running the Protocol

def run_bb84_plain(num_qubits=100, sample_size=20, threshold=0.20):
    # Print Header
    print("BB84 without attacker")
    print("=" * 50)

    # Alice
    # Alice generates random secret bits using quantum randomness
    alice_bits = quantum_random_bits(num_qubits)

    # Alice randomly chooses a basis for each bit
    # 0 = computational basis, 1 = diagonal basis
    alice_bases = quantum_random_bits(num_qubits)

    # Alice prepares one qubit for each bit and basis choice
    qubits = [prepare_qubit(bit, basis) for bit, basis in zip(alice_bits, alice_bases)]

    # Confirm that Alice has created the qubits to send to Bob
    print("Alice prepared", num_qubits, "qubits.")

    # Bob
    # Bob randomly chooss which basis to use for measuring each qubit
    bob_bases = quantum_random_bits(num_qubits)

    # Bob measures each qubit using chosen basis
    bob_bits = [measure_qubit(qubit, basis) for qubit, basis in zip(qubits, bob_bases)]

    # Confirm that Bob has measured all the qubits
    print("Bob measured the qubits.")

    # Public basis comparison and key sifting
    # Aice and Bob publicly comapre their bases, not their bit values
    # Keep positions that uses same basis
    matching_indices = [i for i in range(num_qubits) if alice_bases[i] == bob_bases[i]]

    # Alice sifted key contains only the bits where bases match
    alice_sifted_key = [alice_bits[i] for i in matching_indices]

    # Bob sifted key contains his measurement results at same position
    bob_sifted_key = [bob_bits[i] for i in matching_indices]

    # Show how many qubits survived the basis comparison step
    print("Number of matching bases:", len(matching_indices))

    # Display both sifted keys so they can be compared in the no attacker case
    print("Alice sifted key:", bit_string(alice_sifted_key))
    print("Bob sifted key:  ", bit_string(bob_sifted_key))

    # Test part of the sifted key for errors
    # If not enough sifted bits, program cannot test the chosen sample size
    if len(alice_sifted_key) < sample_size:
        print("Not enough matching bits for the chosen sample size.")
        return [], []

    # Alice reveals the first part of her sifted keys for testing
    alice_test = alice_sifted_key[:sample_size]

    # Bob reveals te corresponding part of his sifted keys
    bob_test = bob_sifted_key[:sample_size]

    # Count how many revealed test bits are different
    errors = sum(1 for a, b in zip(alice_test, bob_test) if a != b)

    # Calculate the proportion of tested bits that do not match
    error_rate = errors / sample_size

    # Display the results of the error test
    print("Testing first", sample_size, "sifted bits.")
    print("Errors:", errors)
    print("Error rate:", round(error_rate, 3))
    print("Detection threshold:", threshold)

    # If the error rate is too high, Alice and Bob assume an attack or noise occurred
    if error_rate > threshold:
        print("Attack/noise detected. Key discarded.")
        return [], []

    # Final Key
    # Remove the test bits because they were publicly revealed
    final_alice_key = alice_sifted_key[sample_size:]
    final_bob_key = bob_sifted_key[sample_size:]

    print("No attack detected. Key accepted.")

    # Display the final secret keys
    print("Final Alice key:", bit_string(final_alice_key))
    print("Final Bob key:  ", bit_string(final_bob_key))

    # Return both final keys so they can be checked or used later
    return final_alice_key, final_bob_key

In [ ]:
# Expected Result

final_alice_key, final_bob_key = run_bb84_plain(
    num_qubits=100,
    sample_size=20,
    threshold=0.20
)